# Old French POS tagger using contextual embeddings

This was used as a formative exercise in JFL484H1 and may be again. Please do not distribute.


In [1]:
import urllib.request
import tarfile
import os

# Download the BERTrade-tiny model from Zenodo
urllib.request.urlretrieve(
    "https://zenodo.org/records/6461220/files/bertrade-tiny-8192_tok+mlm-fro-32e.tar.xz?download=1",
    "bertrade-tiny-8192_tok+mlm-fro-32e.tar.xz")
tarfile.open("bertrade-tiny-8192_tok+mlm-fro-32e.tar.xz", "r:xz").extractall()
os.remove("bertrade-tiny-8192_tok+mlm-fro-32e.tar.xz")

# Download the split Profiteroles data from Github
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Old_French-PROFITEROLE/refs/heads/master/fro_profiterole-ud-train.conllu",
    "fro_profiterole-ud-train.conllu")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Old_French-PROFITEROLE/refs/heads/master/fro_profiterole-ud-dev.conllu",
    "fro_profiterole-ud-dev.conllu")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Old_French-PROFITEROLE/refs/heads/master/fro_profiterole-ud-test.conllu",
    "fro_profiterole-ud-test.conllu")

%pip install pandas
%pip install torch
%pip install msgpack
%pip install zstd
%pip install transformers

/tmp/ipykernel_79581/2865861624.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tarfile.open("bertrade-tiny-8192_tok+mlm-fro-32e.tar.xz", "r:xz").extractall()



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Switching from Pytorch to Transformers

In the demo in Week 9, we demonstrated Pytorch, a library for fitting neural network models. For this exercise we need to use Transformers, a library that wraps Pytorch and adds some features. Don't get confused: "transformers" are a type of neural network architecture we talked about in LEC. But "Transformers" (capital T) is a library.

The BERTrade model was saved using this library, and it takes care of several things that Pytorch does not.

- **Architecture and naming.** Uses a file called `config.json` to store the model architecture and includes a lot of logic to support common variants on how the components of the model are named.

- **Tokenization.** Uses a file called `tokenizer.json` to set up a tokenizer.

- **Tweaked neural net operations.** Although we didn't talk about them, BERT uses two features called layer normalization and GELU activation. Pytorch can do them, but BERT does them in a very specific way that is easier to set up in Transformers.

We will need to load the BERTtrade model using the Transformers library. After that, more or less everything is as in Pytorch, with a few minor tweaks that we'll flag.

Transformers was already installed for you in the setup cell above, and we also downloaded the model. We're going to use the "tiny" version to make sure that you can actually fit it in the limited memory you have on JupyterHub. If you poke around in your files, you'll see that after running the setup cell, there was a directory created called `bertrade-tiny-8192_tok+mlm-fro-32e` which contains the following files:

- `config.json` - specifies the model architecture
- `pytorch_model.bin` - the parameters of the trained BERTrade model
- `vocab.json` - the vocabulary: a mapping from character strings to integer codes
- `tokenizer_config.json` - the settings for the "tokenizer" preprocessing step that we're going to use (discussed further below)
- `special_tokens_map.json` - information about special tokens like `<s>`
- `merges.txt` - instructions for the BPE algorithm on how (in particular in what order) to merge characters into tokens

Loading the model and its tokenizer is as simple as running the following code:

In [2]:
from transformers import AutoTokenizer, AutoModel

model_dir = "bertrade-tiny-8192_tok+mlm-fro-32e"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
bertrade_base = AutoModel.from_pretrained(model_dir)

/home/emd/W/Teaching/JFL484 2026/JFL484-preparation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 37/37 [00:00<00:00, 19885.86it/s]
RobertaModel LOAD REPORT from: bertrade-tiny-8192_tok+mlm-fro-32e
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    |

This report confirms that the model loaded successfully. The 'UNEXPECTED' keys are just leftover parts from the masked language modelling setup (which we don't need), and the 'MISSING' keys are parts of the standard BERT structure for collapsing across tokens that we aren't using anyway.

In [3]:
print(bertrade_base)

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(8192, 128, padding_idx=1)
    (token_type_embeddings): Embedding(1, 128)
    (LayerNorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 128, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-1): 2 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=128, out_features=128, bias=True)
            (key): Linear(in_features=128, out_features=128, bias=True)
            (value): Linear(in_features=128, out_features=128, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=128, out_features=128, bias=True)
            (LayerNorm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (dropout):

Have a look at the output of the previous code cell. It lists the architecture of the "tiny" model. There are several things in here you will not yet understand:  **dropout,** **layer norm,** **GELU,** and the **pooler.** GELU is an alternative to the logistic function---this is all you need to know about that. We'll go over the others briefly below. The pooler is important and we'll need to deal with it as well. 

The `RobertaSelfOutput` layer is hiding something we talked about: the residual connection. Although it isn't stated explicitly in this printout, it's this layer that takes the output of the value matrix and adds it to the initial input. (In fact, this is also the case for the `RobertaOutput` layer at the end of the feed-forward block.)


**Your task.**

In the following markdown cell, explain what each of the `in_features`, and `out_features` values inside the `RobertaSelfAttention` block as well as the `RobertaIntermediate` block means (note that there is more than one of each), relating them back to the explanation of transformers in lecture/the textbook. It will also be useful to examine the BERTrade paper to make sure everything you're seeing can be found explained in the paper. If you get stuck, it may also be worth looking back at the demo notebook from Week 9.


**Solution.**

`(query): Linear(in_features=128, out_features=128, bias=True)`

This means that the query vectors have 128 variables (`out_features`). They take 128 values as input (`in_features`) because that's what both the vocabulary embeddings and the previous transformer layer output.

`(key): Linear(in_features=128, out_features=128, bias=True)`

The key vectors need to match the query vectors, so it makes sense that they have the same `out_features`. The `in_features` are the same because both are drawing on the same inputs.

`(value): Linear(in_features=128, out_features=128, bias=True)`

The value matrix determines what the first phase of the self-attention block is actually contructing (128 values as indicated in `out_features`). Once again, it needs to have the same `in_features` as the query and value matrices because it takes the same features as input.

The `RobertaIntermediate` layer is as follows:

`(dense): Linear(in_features=128, out_features=512, bias=True)`

This means that the feed-forward layer is actually mapping the values up to a higher dimension (512), before mapping them back down to 128 in `RobertaOutput`.

## Making a plan to build a POS tagger on top of BERTrade

Your goal is to build a part-of-speech tagger. That is, a model that takes words as input and predicts (classifies) their part of speech. You certainly have a lot of words, and each does indeed have a part-of-speech label in the UPOS field. The general idea is that you will pass your words into BERTrade, let BERTrade compute representations for them, and modify BERTrade so that the representations go into a classifier. However, there are a few things to think about here, which are different from our setting in the Week 9 demo, where we had one word as input, and one class (also a word) as output.

- **You need to pass in a whole utterance,** not an indvidual word. BERT representations are contextual. Thus, it is critically important that you actually _include_ the context.
- **Remember that BERT maps tokens to (transformed) tokens.** The representations go through the transformer layers and come out as another sequence with the same length as the original, but in a transformed space.
- **Each word in the utterance has a part of speech tag.** While it's true that you _could_ treat each word as a separate data point, and load it separately in the model, if you are already passing in the whole context (i.e., the whole sentence), shouldn't there be a way to tag all the words at once?
- **Tokenization.** BERTrade's inputs are BPE tokens. There will in general be several BPE tokens per word. Thus, although your inputs will be subword tokens, somehow, you need to make sure that your outputs are words.
- **Pooling.** Currently you will have noticed that BERTrade has a "Pooler" layer at the end. "Pooling" means (roughly) averaging: averaging over _all_ the tokens in the sentence in order to get one single representation of the whole sentence. Although that might be useful in some contexts (such as classifying an entire sentence), that is not what we want here. We'll need to remove or bypass that layer. (We say "pooling" instead of "averaging" for a reason: taking the mean is not the only way to pool, and, in many contexts, we may wish to do other operations, such as taking the maximum, in order to aggregate down to a single vector.)
- **Freezing.** Winter is coming! We'll do the frozen feature approach rather than full fine-tuning. Therefore, we need to make sure that we are _only_ training the classifier, and not re-training BERTrade.
- **Held-out testing data.** Up to now we have been training and evaluating on the same data. We should probably not do that, as it does not really measure what we usually want to measure: performance on new data.

## Reading and tokenizing

With all this in mind, let's read in the data. We are going to get it from another source. Our own version of the data had French and English translations, but we did not split into training data and testing data: we used the whole PROFITEROLE corpus. The original PROFITEROLE corpus, on the other hand, came with a pre-defined split into training data and testing data. It is important to use existing splits wherever possible, so that your results are comparable with published resultas in the literature.

The above suggests that we should read each sentence in as a sentence. The fact that the tokenization is done _for us_ using a BPE model means that we should **not** try and convert our text into tokens, and certainly not into integers.

However, we will need to extract, for each sentence, the sequence of UPOS tags. For the moment, let's leave them as strings.

The original PROFITEROLE data set has been downloaded for you. There are three files in your current working directory, each of which are single CONLL-U files that combine sentences from many texts:

- `fro_profiterole-ud-train.conllu` - Standard training set 
- `fro_profiterole-ud-test.conllu` - Standard test (evaluation) set
- `fro_profiterole-ud-dev.conllu` - Standard development set

We'll talk about the development set shortly.

There is a function below which reads in the data from a CONLL-U file. It returns :

- A list of sentences, each sentence being a list of words (strings)
- A list of lists of UPOS tags, one for each word in the corresponding sentence
- A list of document id's, one per sentence
- A list of sentence id's, one per sentence

We won't be using the document or sentence id's in this exercise, but in general (and in Homework 2) it may be useful to have some meta-data like this. 

In [4]:
def read_conllu(fn):
    with open(fn, encoding="utf-8") as hf:
        sentences = []
        sentences_pos = []
        sentences_doc = []
        sentences_sentid = []
        words = []
        pos = []
        doc = None
        sent_id = None
        for line in hf:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith("# text"):
                if len(words) > 0:
                    assert len(pos) == len(words)
                    assert sent_id is not None
                    sentences.append(words)
                    sentences_pos.append(pos)
                    sentences_doc.append(doc)
                    sentences_sentid.append(sent_id)
                words = []
                pos = []
                doc = None
                sent_id = None
                continue
            if line.startswith("# newdoc id"):
                doc = line.split(" = ")[1]
                continue
            if line.startswith("# sent_id"):
                sent_id = line.split(" = ")[1]
                continue
            if line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) not in (8, 10):
                continue
            try:
                _ = int(parts[0])
            except ValueError:
                # skip multiword tokens like "1-2" or empty-node IDs like "1.1"
                continue
            words.append(parts[1])
            pos.append(parts[3])
        if len(words) > 0:
            assert len(pos) == len(words)
            assert sent_id is not None
            sentences.append(words)
            sentences_pos.append(pos)
            sentences_doc.append(doc)
            sentences_sentid.append(sent_id)
        words = []
        pos = []
        doc = None
        sent_id = None
    return sentences, sentences_pos, sentences_doc, sentences_sentid


The cell below demonstrates some of the basic functionality of the Transformers tokenizer. 

**Your task:** Explain in your own words what the `tokenizer` does, integrating **everything** you see in the cell below. Be precise. You are welcome to add additional code and look in the documentation for Transformers tokenizers to explore what is going on. There is one strange character `'Ġ'` which you are likely to _not_ fully understand. Do your best at describing what you can observe about it. We will be discussing it shortly.

In [5]:
sentences, pos_tags, _, _ = read_conllu("fro_profiterole-ud-dev.conllu")

In [6]:
print(sentences[0])

# Tokenize the first sentence using method #1 - split into words
tokens_v1 = tokenizer(sentences[0], is_split_into_words=True)
print(tokens_v1['input_ids'])
print(tokenizer.convert_ids_to_tokens(tokens_v1['input_ids']))

# Tokenize the first sentence using method #2 - words joined back together with spaces
tokens_v2 = tokenizer(" ".join(sentences[0]), is_split_into_words=False)
print(tokens_v2['input_ids'])
print(tokenizer.convert_ids_to_tokens(tokens_v2['input_ids']))

['Carles', 'li', 'reis', ',', 'nostre', 'emperere', 'magnes', ',', 'Set', 'anz', 'tuz', 'pleins', 'ad', 'estet', 'en', 'Espaigne', ':']
[0, 869, 437, 461, 5861, 16, 4195, 3284, 385, 1225, 3251, 16, 55, 274, 495, 88, 491, 1210, 484, 628, 291, 274, 262, 5390, 30, 2]
['<s>', 'Car', 'les', 'li', 'reis', ',', 'nostre', 'emper', 'ere', 'ma', 'gnes', ',', 'S', 'et', 'anz', 't', 'uz', 'ple', 'ins', 'ad', 'est', 'et', 'en', 'Espaigne', ':', '</s>']
[0, 869, 437, 325, 1261, 389, 454, 5812, 357, 3251, 389, 419, 274, 1956, 1589, 773, 484, 493, 309, 274, 293, 1263, 2036, 666, 2]
['<s>', 'Car', 'les', 'Ġli', 'Ġreis', 'Ġ,', 'Ġnostre', 'Ġemperere', 'Ġma', 'gnes', 'Ġ,', 'ĠS', 'et', 'Ġanz', 'Ġtuz', 'Ġple', 'ins', 'Ġad', 'Ġest', 'et', 'Ġen', 'ĠEs', 'paigne', 'Ġ:', '</s>']


**Solution.**


## More about tokenization

BERTrade uses the same tokenizer as RoBERTa, BPE. You can see the tokenizer configuration in the file `tokenizer_config.json`. Let's review. It will be important to read this before moving on.

### UNK, BOS, EOS
```
{"unk_token":
    {"content": "<unk>", "single_word": false, "lstrip": false, "rstrip": false, "normalized": true, "__type": "AddedToken"},
"bos_token":
    {"content": "<s>", "single_word": false, "lstrip": false, "rstrip": false, "normalized": true, "__type": "AddedToken"}, 

"eos_token":
    {"content": "</s>", "single_word": false, "lstrip": false, "rstrip": false, "normalized": true, "__type": "AddedToken"}, 
```

This is the configuration for "unknown tokens" as well as for the bOS and EOS tokens. The whole point of subword tokenization is to eliminate unknown tokens, but it can happen if it was not set up to handle all the actual characters in the text. If so, it simply replaces them with a special token `"<unk>"`. 

To understand the remaining parameters (`single_word`, `lstrip`, `rstrip`, `normalized`) we need to understand **how** these special tokens get introduced during the tokenization process. In particular, they are "injected" directly into the original string BEFORE any tokenization takes place. Only then is it split into tokens. We won't go into detail of these parameters, but, on the whole, they control how the special tokens get extracted from the text, to ensure that the process matches the way they are injected.

### Prefix spaces
```
"add_prefix_space": false,
```
**This is important.** There is a choice to be made when training a sequence model, as to how to deal with sentence-initial words. Although there is already a `'<s>'` token, sometimes sequence models make the choice to nevertheless treat sentence-initial tokens as different tokens than sentence-medial ones. The way that BERTrade does this is by prefixing a special character `'Ġ'`---the mysterious character we encountered above, which it calls a "space" character---to all tokens EXCEPT for the first word. This is the parameter that tells it not to add it to the first word. Thankfully, this character never actually appears in Old French. (It _is_ used in traditional Irish orthography, so we would need to change what the "space" character is if we were working with Old Irish to avoid conflicts.) **This is going to cause some issues which we will come back to shortly.**


### Errors
```
"errors": "replace",
```
This says that the tokenizer will replace ill-formed characters with a special "replacement" character. This isn't the same thing as `"<unk>"`. This replacement character is for actual errors in the encoding of the text, such that the sequence of bytes is simply not a valid Unicode string. The `"<unk>"` symbol, on the other hand, is for valid characters which happen not to be in the BPE dictionary (such as a Chinese character or an emoji, which were never encountered in the original corpus and therefore are not handled by the tokenizer).


### SEP and CLS
```
"sep_token":
    {"content": "</s>", "single_word": false, "lstrip": false, "rstrip": false, "normalized": true, "__type": "AddedToken"}, 

"cls_token":
    {"content": "<s>", "single_word": false, "lstrip": false, "rstrip": false, "normalized": true, "__type": "AddedToken"}, 
```
These lines deal with two special tokens used in the original BERT. As we mentioned very briefly in class, the original BERT's pre-training was not _just_ masked language modelling, but had an additional phase in which pairs of sentences were input, and the model had to determine whether they went together in the original text. The `sep_token` was used to separate the two sentences. It's not necessary here, but the setup of the tokenizer still wants to be told how to deal with it. We set it to the EOS token, but it will never be used.

The `cls_token` was a token that was prepended to the beginning of every sentence in BERT. It was used in sentence-level classification tasks such as sentiment classification to be "the guy": since it sucks in context from the rest of the sentence, it will be useful for getting information from across the entire sentence. It was convenient to put it at the beginning so that it would have a fixed position (0). However, if we are only ever feeding in one sentence at a time, this also becomes redundant with BOS. Thus, once again, we tell the tokenizer they are the same token, and we never actually use the CLS token.

### PAD and MASK
```
"pad_token":
    {"content": "<pad>", "single_word": false, "lstrip": false, "rstrip": false, "normalized": true, "__type": "AddedToken"}, 

"mask_token":
    {"content": "<mask>", "single_word": false, "lstrip": true, "rstrip": false, "normalized": true, "__type": "AddedToken"},
``` 
The `"<pad>"` and `"<mask>"` tokens are the tokens the model uses to pad shorter sentences (see below) and to mask out tokens for the masked prediction task (i.e., the tokens that need to be predicted), respectively.

### Window size
```
"max_len": 512,
```
The maximum sequence window size, in terms of total number of tokens, including BOS and EOS. The tokenizer will actually allow you to go way over length: it will simply spit out a warning the first time you go over the limit. However, such inputs won't fit in the model. If you had any (we don't here), then you would have to think of a plan as to what to do with them.

### Additional configuration
```
"special_tokens_map_file": null,
"name_or_path": "local/tokenizers/bertrade-8192"}
```
(Only for the curious.) It's possible to put the information about BOS, EOS, SEP, etc in a separate file, rather than in the main config file. Although this file does exist here, it's only for reference. All of this is contained in the main config file. This is why `special_tokens_map_file` is set to `null`. The other field is just the name of the local file where the researcher originally stored the tokenizer configuration while they were working on the model.


## Mapping words to tokens

As we just saw, the tokenizer makes an important distinction between initial and medial words. However, as you saw above, it _only_ makes this distinction when we pass the entire sentence in as a giant string (**version 2**), and not when we pass it in as a list of words (**version 1**). The BERTrade model, however, was trained on raw text. This means that, if we don't have the `Ġ`'s in our input, to BERTrade, all our words will look like they 

One possibility would be to use **version 1** and to manually fudge the output to paste the `Ġ` character back into our tokens. Although we will wind up needing to do something kind of like this anyway, I would encourage you to be somewhat nervous about messing with the output of the tokenizer do when you're still learning. Let's  use **version 2.** 

This is quite annoying. Your job is to build a **POS tagger.** POS tags are assigned to words---not BPE tokens. Although we haven't dealt with it yet, at some point in your pipeline, you are going to have to take the output of last transformer layer and **merge the (potentially multiple) BPE tokens for each word** back into a single representation for that word. By joining the sentence together with spaces, we are going to break your only way of figuring out which token goes with which word in the original CONLL-U file. Mighty frustrating, isn't it? 

Thankfully, modern versions of Transformers tokenizers can **keep track of where each token came from in the original string** (sort of). This will help us reconstruct which word each token belongs to.

We need to turn this feature on, but when we do, we obtain a new entry in the dictionary. Let's have a look. 

In [7]:
tokens_v3 = tokenizer(" ".join(sentences[0]),
                      is_split_into_words=False,
                      return_offsets_mapping=True)

print(tokens_v3['offset_mapping'])
print(len(tokens_v3['offset_mapping']))

print(tokenizer.convert_ids_to_tokens(tokens_v3['input_ids']))
print(len(tokens_v3['input_ids']))

[(0, 0), (0, 3), (3, 6), (7, 9), (10, 14), (15, 16), (17, 23), (24, 32), (33, 35), (35, 39), (40, 41), (42, 43), (43, 45), (46, 49), (50, 53), (54, 57), (57, 60), (61, 63), (64, 67), (67, 69), (70, 72), (73, 75), (75, 81), (82, 83), (0, 0)]
25
['<s>', 'Car', 'les', 'Ġli', 'Ġreis', 'Ġ,', 'Ġnostre', 'Ġemperere', 'Ġma', 'gnes', 'Ġ,', 'ĠS', 'et', 'Ġanz', 'Ġtuz', 'Ġple', 'ins', 'Ġad', 'Ġest', 'et', 'Ġen', 'ĠEs', 'paigne', 'Ġ:', '</s>']
25


This `offset_mapping` is a list of tuples: for each token in the BPE token list (note that they have the same length), the beginning and and end of the token in the input, in Python slice format (beginning index, 1+end index). `(0,0)` is a special value that indicates an inserted token. While this doesn't tell us where the word boundaries are, it should help us.

Let's verify that the first few "real" tokens do indeed map correctly back to the joined string we passed in.

In [8]:
print(" ".join(sentences[0])[0:3])
print(" ".join(sentences[0])[3:6])
print(" ".join(sentences[0])[7:9])
print(" ".join(sentences[0])[10:14])

Car
les
li
reis


Since we know exactly where the spaces are that we inserted into the string when we `join()`ed (before every word except the first), we can straightforwardly reconstruct the beginning and end index of each of the original words we pulled out of the corpus. 

Remember also that BPE tokens are always calculated _within_ words. What does "word" mean for the tokenizer given that we pass in a single string where we joined the words together? Well, the tokenizer starts with a first-pass **preliminary tokenization** which involves breaking on spaces (among other things). Then, BPE tokens are calculated within each of these preliminary tokens.

The consequence of this is that a single BPE token **will never span a space** in the string we passed in. Therefore, we should never have any alignment issues whereby a single BPE token represents multiple words.

**Your task.** Write a function called `align_tokens_to_words()` which takes as input:

- A sentence as a list of words
- The token-to-character offset mapping from a tokenization of the sentence (so, the value of the `offset_mapping` entry of the tokenization as explored above)

It should return **an alignment of words to tokens.** The alignment should be in the same format as the token--character alignment that we just saw---a list of tuples---but it should represent something different. Instead of having one tuple per token, it should have one tuple per **word** in the sentence. And, instead of representing slices of a character string, each tuple should represent a **slice into the list of tokens,** corresponding to the span of tokens that make up the given word.

Throw an exception if there is a misalignment (a word boundary doesn't correspond cleanly to any token boundary).

In [ ]:
## SOLUTION


TEST YOUR CODE in the following cell. There is no need to write a formal test that verifies an output against an expected output, but your test must demonstrate clearly to you, and to us, that your code functions correctly.

In [ ]:
## Solution - just print the alignment for one sentence - should be enough


['Car', 'les']
['Ġli']
['Ġreis']
['Ġ,']
['Ġnostre']
['Ġemperere']
['Ġma', 'gnes']
['Ġ,']
['ĠS', 'et']
['Ġanz']
['Ġtuz']
['Ġple', 'ins']
['Ġad']
['Ġest', 'et']
['Ġen']
['ĠEs', 'paigne']
['Ġ:']
['Carles', 'li', 'reis', ',', 'nostre', 'emperere', 'magnes', ',', 'Set', 'anz', 'tuz', 'pleins', 'ad', 'estet', 'en', 'Espaigne', ':']


## Padding

One more remark needs to be made here. The tokenizer outputs a dictionary, and we have not looked at the third entry, which is called `attention_mask`.

This is related to the padding tokens we discussed in class that fill up the empty spaces in shorter sentences. The attention mask is what tells the model not to pay attention to the padding tokens. It is a list of the same length as the list of tokens, which contains `0` for any padding token, and `1` for the non-padding tokens.

So far, we haven't told the tokenizer to do any padding. Therefore, there are no padding tokens and if we examine the attention mask, we see all 1's.


In [11]:
print(tokens_v3['attention_mask'])

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


### Why pad?

Before activating padding, let's consider when we actually need it. In class, we said that padding was necessary to deal with the fact that "sequences have variable lengths." We brought this up in the context of hypothetical feed-forward networks wherein the actual **architecture of the network**  would depend on the length of the input.

However, we did not return to the question of padding in detail the context of self-attention: all we said is that we still needed a window size, and that large language models were large in part because they have large windows. But, importantly, in self-attention, the weights applied to each token are the same. Therefore, the same issue does not apply: the size of the network does not rely on the input sequence having any fixed length.

### Matrix multiplication

The reason that we need to have a predefined window size in a self-attention network is different in an important way. The operations of calculating **query,** **key,** and **value** vectors
can all be parallelized across multiple input sequences.

If we have a single sequence of inputs, the operation of combining the weights with the inputs and taking the sum to calculate each of the query (or key, or value) vectors for each token can be done by taking the weights associated with each individual query variable and taking the dot product with respect to each input vector. (The dot product is the basic operation of taking two vectors of the same size, multiplying each corresponding pair of elements, and taking the sum.) Those input vectors can be arranged in a matrix with $d$ rows (where $d$ is the number of input variables) and $N$ columns (where $N$ is the length of the sequence). Calculating the dot product of the weight matrix over each of the input vectors is a **matrix multiplication.** This can be done quickly in parallel on a GPU (and somewhat less quickly in parallel on a computer without a GPU).

Nothing up to now requires that the sequence have any particular length. What is important is only that the sequence have a **maximum length** in order to ensure that it fits in the memory of the GPU. The same is true for the calculation of the attention weights.

### Padding and tensor multiplication

The constraint arises when we attempt to further parallelize to deal with multiple input sequences. We can arrange our input sequences, each of which is a matrix, into three-dimensional array called a **tensor.** However, while it is technically possible to construct such an array out of matrices representing sequences of different lengths (different $N$---a "ragged array"), this is not something that GPUs are designed to do efficiently.

This is why we pad in transformer models: so that, **across all sequences that we are passing in in parallel,** all sequences have the same length. However, there is no reason to pad sequences to the maximum possible length. Since we will be training in mini-batches, the only important thing is that, within each batch, the sequences be padded **to the length of the maximum sequence.**

More recently, many models have switched from representing batches as tensors to doing **sequence packing:** concatenating separate sentences into a single long sequence of fixed (maximum) length, and passing it in as a single matrix. However, this requires some bookkeeping that we won't deal with here.

### The attention mask

Returning to the attention mask, its role is simple: in calculating the softmax over the attention weights, multiply the probabilities assigned to  the padding tokens by zero (or, rather, a very very small number close to zero). 

### Padding in practice

**Let's examine how padding will actually work** by passing in two sentences to the tokenizer. Note that the tokenizer is happy to accept a list of sentences, and knows what to do (tokenize each sentence). It can then accept an argument asking it to pad to the maximum length.

In [12]:
tokens_v4 = tokenizer([" ".join(s) for s in sentences[0:2]],
                      is_split_into_words=False,
                      return_offsets_mapping=True,
                      padding=True)
print("Sentence 1 tokens: ", tokenizer.convert_ids_to_tokens(tokens_v4['input_ids'][0]))
print("Sentence 2 tokens: ", tokenizer.convert_ids_to_tokens(tokens_v4['input_ids'][1]))
print("Sentence 1 attention mask: ", tokens_v4['attention_mask'][0])
print("Sentence 2 attention mask: ", tokens_v4['attention_mask'][1])

Sentence 1 tokens:  ['<s>', 'Car', 'les', 'Ġli', 'Ġreis', 'Ġ,', 'Ġnostre', 'Ġemperere', 'Ġma', 'gnes', 'Ġ,', 'ĠS', 'et', 'Ġanz', 'Ġtuz', 'Ġple', 'ins', 'Ġad', 'Ġest', 'et', 'Ġen', 'ĠEs', 'paigne', 'Ġ:', '</s>']
Sentence 2 tokens:  ['<s>', 'Tres', 'qu', "'", 'Ġen', 'Ġla', 'Ġmer', 'Ġcun', 'quist', 'Ġla', 'Ġtere', 'Ġal', 't', 'aigne', 'Ġ.', '</s>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
Sentence 1 attention mask:  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Sentence 2 attention mask:  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]


From this, we see that the second sentence is shorter than the first. It is filled with padding tokens, and its attention mask is set to zero for these padding tokens.

## Testing your word alignment code

You wrote your token-to-word aligment function before you knew there were going to be padding tokens to deal with.

**Your task.** Test that your `align_tokens_to_words()` function still works with padded sentences. You can access the offset mapping of the padded version of `sentences[1]` via `tokens_v4['offset_mapping'][1]`.


In [ ]:
## SOLUTION

['Tres', 'qu', "'"]
['Ġen']
['Ġla']
['Ġmer']
['Ġcun', 'quist']
['Ġla']
['Ġtere']
['Ġal', 't', 'aigne']
['Ġ.']
["Tresqu'", 'en', 'la', 'mer', 'cunquist', 'la', 'tere', 'altaigne', '.']


## Writing the code to prepare the data

There are two steps remaining before we can get training underway. The last thing we do will be to adapt the BERTrade model to do POS classification. However, the first and more fundamental step will be to write the code that converts the data---both the inputs and the outputs---into a format that the model can deal with. 

We have now seen a preview of several issues that are going to arise:

- Tokenization
- Padding the sequences within a batch
- Matching the words to the tokens

We are going to be working with mini-batches, and each mini-batch will be specified as a set of (randomly drawn) indices into the list of sentences/corresponding POS tags. As such, what we want is a piece of code which, given a set of indices, will return a tensor object containing, on the one hand, the padded sequences in the mini-batch, and, on the other, the POS tags.

Given the way Pytorch/Transformers work, we are going to do this using three layers:

- A custom **DataSet** class: a bundle of code that specifies how to prepare **one single sentence** (both inputs and POS labels)
- A **data collator:** a function that processes multiple sentences and packages them together appropriately into a single tensor for the inputs and a single tensor for the labels
- A **DataLoader:** generates random mini-batches, calls the DataSet code for each of the items in the batch, and compiles them using the data collator

The DataLoader will be taken off-the-shelf---no need for custom code. The data collator will also be a very simple one that already exists in Transformers called `DataCollatorForTokenClassification`, which takes a list of sequences and corresponding labels and deals with padding (more details to come).

Thus, the only thing we will need to write is the code for the **DataSet** class. However, before we can do this, we will need to make one more decision.


### Dealing with the token--word alignment

Before we can start writing the code to load the data, we need to clarify one last detail. Although we have a correspondence between tokens and words, we still don't know exactly how we are going to carry out our POS tagging.

For each sentence, our transformer is going to generate a sequence of vectors, one for each token in the sentence. Let's say there are 25 tokens. Now let's say that corresponds to 17 words, each of which has a POS tag. What we want is therefore 17 softmax regressions. How do we get this out of 25 tokens?

There are a few ways we could do this:

- **Averaging.** Use our token-word correspondence to average the token vectors within each word.
- **Max-pooling.** As with averaging, but instead of taking the average value of each dimension, take the maximum. This is a common alternative to averaging.
- **One token per word.** Copy the POS tags out so that the same POS tag is repeated for every token in a given word. Then take only one token as a "representative" of the whole word: pass _all_ tokens through a softmax classifier, but somehow ignore all but one of the tokens of a given word when calculating the loss function.

The first two approaches have the disadvantage that they make it so the sequences in a batch are no longer of equal length. We have at least two options for dealing with this, but they are both complicated. 

Let's instead use the third approach: **one token per word.** 

### Hiding from the loss function

Let's take the first token of the word as its representative.  To see how we can do this, let's first ask how we can get the `CrossEntropyLoss` to ignore certain elements of the sequence.

If we were writing our own loss function, this would be quite easy: we could simply inject some special value into the sequence of POS values for the elements of the sequence we want to ignore. Then, when calculating the loss, we could simply look for this value. We saw in the week 9 demo that, when we are doing softmax classification, we code our output categories as whole numbers (0, 1, 2, ...). Since we would never have any use for a negative number, we could simply use a negative number as this special value.

It turns out that we don't need to rewrite anything: this is such a common pattern that `CrossEntropyLoss` already does it! We simply need to use the special index -100. The loss function will simply ignore element of the sequence with this index as a label. (We can customize what number the index is with the `ignore_index` keyword argument when constructing the loss function, but we don't really need to do this here.)

This also solves another problem: the beginning-of-sentence, end-of-sentence, and padding tokens. These too should be ignored when calculating the loss. To do so, we simply need to set the POS tag for these to index -100.

### Putting it all together

We therefore are going to construct a function that, given the index of a single sentence:

- Gets that sentence, gets its POS tags
- Tokenizes the sentence (no padding)
- Using the `align_tokens_to_words` function, constructs a sequence of POS tags (as numerical indices) of the same length as the tokenized sentence, where every element is -100 except for the first token of each word

We'll have to build (outside the function) a mapping from POS tags to numerical indices. We'll deal with padding later.

**Here is the code that will do this** (with, of course, some important parts missing, which will be for you to write). We create a custom class which we call `POSDataset`. The `__getitem__` method is the code that accesses individual items. 

In [14]:
from torch.utils.data import Dataset

class POSDataset(Dataset):
    def __init__(self, sentences, pos, tokenizer, pos_ids):
        self.sentences = sentences # List of lists of words
        self.pos = pos             # List of lists of POS tags
        self.tokenizer = tokenizer
        self.pos_ids = pos_ids

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        pos_tags = self.pos[idx]

        ####
        ## YOUR CODE HERE
        ####

        ### Solution
        tokens = self.tokenizer(
            " ".join(words),
            is_split_into_words=False,
            return_offsets_mapping=True
        )
        alt = align_tokens_to_words(words, tokens['offset_mapping'])
        labels = [-100]*len(tokens['input_ids'])
        for i, a in enumerate(alt):
            labels[a[0]] = self.pos_ids[pos_tags[i]]
        ###
        
        return {
            'input_ids': tokens['input_ids'],
            'attention_mask': tokens['attention_mask'],
            'labels': labels
        }

We have given you the requisite structure. The above code creates a class (essentially a template for making complex objects) which is a subclass of the Pytorch `Dataset` class. 

If your object-oriented Python experience is limited or rusty, you should note a few things:

- The `__init__` method is the constructor, the function that is called when we create a new instance of the class by calling `POSDataset()`
- The functions defined inside the class (methods) can be called by doing `<object>.<method_name>()` where `<object>` is an instance of the class
- Methods have an obligatory first argument which is  `self`, but we don't actually pass this in ourselves
- We'll never actually be calling `__getitem__` manually. Our `DataLoader` will be calling it for us when it wants to retrieve a new item. It will be passing in the argument `idx`, which is the index of the item we want to retrieve

The data collator (which we'll be making use of later) needs to output tensors for the tokens (encoded as their integer id's), the attention mask, and for the POS labels. We pass these in as dictionaries.


**Your task.** Fill in the above function so that it returns the token id's, the label . Again, no padding should happen at this point---the data collator will take care of that.

Assume that the data, the tokenizer, and a POS-tag index dictionary will have been passed in via the constructor. They are accessible via `self.sentences`, `self.pos_tags`, `self.tokenizer` and `self.pos_ids`.

The code below should serve as a simple test that your code works. **However, it is your job to examine the output and make sure that it corresponds.**

In [15]:
unique_pos_tags = set()
for p in pos_tags:
    unique_pos_tags |= set(p)
pos_ids = dict(zip(
    unique_pos_tags,
    list(range(len(unique_pos_tags))))
)
pos_ds = POSDataset(
    sentences,
    pos_tags,
    tokenizer,
    pos_ids
)

## Should return the second sentence
item_1 = pos_ds[1]
print(item_1['input_ids'])
print(tokenizer.convert_ids_to_tokens(item_1['input_ids']))
print(item_1['attention_mask'])
print(item_1['labels'])

## For comparison
print(sentences[1])
print(pos_tags[1])
print(pos_ids)


[0, 2855, 276, 11, 293, 299, 608, 1411, 7194, 299, 2331, 426, 88, 1546, 412, 2]
['<s>', 'Tres', 'qu', "'", 'Ġen', 'Ġla', 'Ġmer', 'Ġcun', 'quist', 'Ġla', 'Ġtere', 'Ġal', 't', 'aigne', 'Ġ.', '</s>']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[-100, 5, -100, -100, 5, 14, 13, 4, -100, 14, 13, 3, -100, -100, 12, -100]
["Tresqu'", 'en', 'la', 'mer', 'cunquist', 'la', 'tere', 'altaigne', '.']
['ADP', 'ADP', 'DET', 'NOUN', 'VERB', 'DET', 'NOUN', 'ADJ', 'PUNCT']
{'X': 0, 'PROPN': 1, 'AUX': 2, 'ADJ': 3, 'VERB': 4, 'ADP': 5, 'PRON': 6, 'ADV': 7, 'SCONJ': 8, 'CCONJ': 9, 'NUM': 10, 'INTJ': 11, 'PUNCT': 12, 'NOUN': 13, 'DET': 14}


## Combining with a DataLoader

The following cell shows how we use our new class with the data collator to add padding to the longest sequence element---and also in the labels---and then to build a `DataLoader` object that pulls out random mini-batches.

**Run this cell a few times.** You'll see that the shape of the input tensor will shrink and grow: it is a matrix of 16 (the size of the mini-batch) by $N$, where $N$ is the length of the longest sequence in the batch.

In [16]:
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer, 
    padding=True,
    label_pad_token_id=-100 
)

train_loader = DataLoader(
    pos_ds, 
    batch_size=16, 
    shuffle=True, 
    collate_fn=data_collator 
)

batch = next(iter(train_loader))
print(f"Batch input shape: {batch['input_ids'].shape}") 

Batch input shape: torch.Size([16, 54])


## Setting up the model for training

We are almost ready to write our optimization loop. We simply need to convert our BERTrade model into a POS tagger. **Examine the following code:**

In [17]:
from torch import nn

class BERTradePOSTagger(nn.Module):
    def __init__(self, bertrade_model, num_labels):
        super().__init__()
        self.bert = bertrade_model
        # Freeze the BERT weights 
        for param in self.bert.parameters():
            param.requires_grad = False
        # Use dropout (see discussion)
        self.dropout = nn.Dropout(0.1)
        # Our model has 128 dimensions
        self.classifier = nn.Linear(128, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask)
        # This is a tensor of shape: 
        #   [batch size, sequence length, 128]
        # That is, for each sequence in the batch,
        #   we have N tokens where N is the maximum length,
        #   each of which is represented by 128 variables
        sequence_output = outputs.last_hidden_state 
        
        # Apply dropout (see discussion)
        sequence_output = self.dropout(sequence_output)
        
        # Get logits (input to (implicit) softmax)
        # This is a tensor of shape: 
        #   [batch size, sequence length, num_labels]
        return self.classifier(sequence_output) 

Note a few things:

- We are creating a class. Once it's instantiated as some object, if we simply do `<object>(<batch>)`, it will do the forward pass (i.e., calculate the POS tag predictions as $z$ values) over the batch
- We've frozen the weights of the actual BERTrade model by asserting that they "don't require a gradient." The gradient won't be calculated for them and they won't be updated
- We've added a linear layer on top
- Dropout: what is it? We saw it mentioned before when we printed the model. Dropout is a trick whereby, during the training of a model, we randomly zero out a certain proportion of the inputs (in this case 10% of the inputs coming into the classification layer---it's also used throughout the model). This helps to avoid the model overfitting to exactly the data it has seen.

## Training the model

We are ready to train. One last thing to notice. Above, we read in the **dev** subset. The purpose of the **dev** subset is to have data to "play with" that isn't in the training or the test set. Typically, we would use it to evaluate the model at the stage where we are still exploring what type of model to use. However, we wouldn't train on it: it's too small, and that's not what it's for.

**Your task.**

In the following cell:

- Read in the training data, and re-generate anything you might need from it
- Construct a POSDataset, DataCollatorForTokenClassification, and DataLoader; use a batch size of 16
- Write an optimization loop using Adam with a learning rate of `1e-5` that runs for 100 epochs

You'll want to base your optimization loop on the one in the week 9 demo. 

To pull consecutive batches from the DataLoader, you can simply do

```
for batch in dl:
```

where `dl` is the name of your DataLoader object. This will pull consecutive mini-batches randomly until no more mini-batches are left that haven't been drawn yet.

There are a few additional things to deal with in this setting. As such, **we have included some of the code that would be unfamiliar to you.**

In [ ]:
import torch

## 1. Read the data in, create POSDataset, collator,
#   DataLoader. Call your DataLoader `loader_tr` for
#   compatibility with our code.
## YOUR CODE HERE
## SOLUTION


# 2. Create the model
model = BERTradePOSTagger(bertrade_base, len(pos_ids))
model.train() # Sets a flag to tell the Dropout layers to turn on

# 3. Create the optimizer
## YOUR CODE HERE
## SOLUTION

# 4. Set up the loss function
## YOUR CODE HERE
## SOLUTION

# 5. Train for 100 epochs
for epoch in range(100):
    loss_total = 0
    for batch in loader_tr:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']

        # 5a. Reset the gradients to zero
        optimizer.zero_grad()

        # 5b. Do the forward pass
        #     The forward pass should return a tensor 
        #       [batch size, sequence length, num_labels]    
        logits = model(input_ids, attention_mask=attention_mask)

        # 5c. [Messing with tensors - not obvious!]
        #     CrossEntropyLoss needs the sentences in the
        #     batch to be flattened into one long sequence,
        #     both on the model output side and on the
        #     side of the labels.
        #     Use these!
        logits_flat = logits.flatten(0, 1)
        labels_flat = labels.flatten(0)

        # 5d. Calculate loss, gradient; do updates
        ## YOUR CODE HERE
        ## SOLUTION

    print(f"Epoch {epoch}, loss {loss_total:.4f}")


Epoch 0, loss 2698.6621
Epoch 1, loss 2406.6541
Epoch 2, loss 2183.7438
Epoch 3, loss 2002.4527
Epoch 4, loss 1848.8623
Epoch 5, loss 1716.6933
Epoch 6, loss 1602.8300
Epoch 7, loss 1503.6330
Epoch 8, loss 1416.7315
Epoch 9, loss 1340.6988
Epoch 10, loss 1274.4237
Epoch 11, loss 1217.3330
Epoch 12, loss 1164.1011
Epoch 13, loss 1120.6069
Epoch 14, loss 1081.0006
Epoch 15, loss 1043.2844
Epoch 16, loss 1013.5596


## Making predictions and calculating accuracy

We should now see how good we are at predicting POS tags. In order to do this, we'll need to load the test set and compute a predicted POS tag for each item in the test set. You should also do the same for the training set for comparison. The accuracy on the training set will in general be higher than the accuracy on a held-out test set.

You will find it impractical to use your DataLoader. Instead, you should loop over the data and pass it in one sentence at a time. **However, you will need to convert your items** (both the token_id's and the attention_mask) **to tensors.** Before you do this, however, you must first wrap the token_id and attention_mask lists in square brackets. This creates a singleton list. The idea is to create a "batch of length one".

To find the index of the POS tag with the highest logit ($z$) value for each token in the sequence, use:

```
torch.argmax(logits, dim=-1).squeeze().tolist()
```

But don't forget to filter out the tokens that would normally be marked -100.


**Your task.**

For the training set and the test set (separately), create a **Pandas DataFrame** containing three columns: the word (string), the true part of speech, and the predicted part of speech. Then construct a fourth column which is `1` if the true and predicted POS match, and 0 otherwise.

Use these data frames to calculate overall accuracy, and use a pivot table to show the accuracy broken down by (true) POS tag. Compare train and test.

**Note.** You will probably find that your performance is far short of the performance reported in the paper, even for the tiny model we are using. You can just accept this. This is most likely due to the fact that your model needs to be trained for longer, or that the learning rate needs to be adjusted. It is also possible, since the PROFITEROLE corpus is an expansion of the SRCMF corpus that BERTrade tested on, that the additional files in the PROFITEROLE corpus are **much** harder to tag than the SRCMF corpus. However, this seems unlikely, and the explanation that your training procedure needs tweaking seems much more plausible.

In [ ]:
## SOLUTION

